# 19 — OE per-pixel inversion with albert_mobley_3C_jax (dask engine)

Companion to NB17 (`17_oe_dask_bi`): replaces the bi_jax forward model with
the coupled Albert-Mobley + 3C glint model, keeping the same per-pixel OE
architecture via `oe_engine_optx` tiled through `dask_engine`.

| Layer | NB17 | NB19 |
|---|---|---|
| Forward model | `bi_jax` | `albert_mobley_3C_jax` |
| Solver | `oe_engine_optx` (JAX vmap OE) | `oe_engine_optx` (JAX vmap OE) |
| Image layer | `dask_engine` (all pixels, tiled) | `dask_engine` (all pixels, tiled) |

**Forward model**: `albert_mobley_3C_jax` — coupled water column + surface glint.  
**Free parameters**: `C_0` (phytoplankton), `C_Y` (CDOM), `C_Mie` (mineral
scattering), `g_dd`, `g_dsr`, `g_dsa`, `d_r` (glint) — **7 total**.  
`C_1`–`C_5` (minor phytoplankton) and `C_X` (ISM) are fixed at their prior
values; they are not needed for the glint-correction step.  
**Scene**: Helsinki EnMAP L2A.

### Motivation vs NB12 (OE superpixel)
NB12 uses `superpixel_engine` which back-interpolates retrieved parameters
from ~5000 segment means to all pixels via PCA + kNN.  That blending step
can produce unphysical parameter combinations and negative Rrs.  NB19 inverts
every water pixel independently — no interpolation artefacts, at the cost of
longer wall-clock time.

## 1  Imports & configuration

In [ ]:
import numpy as np
import lmfit
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import time
import xarray as xr
import rioxarray
from pyproj import CRS
from xcube.core.store import new_data_store
import configparser
import jax

from bio_optics.coupled_models import albert_mobley_3C_jax
from bio_optics.inversion import oe_engine, oe_engine_optx
from bio_optics.image_processing import dask_engine

jax.config.update('jax_enable_x64', True)

In [ ]:
NOISE      = 0.0040  # Rrs noise level [sr-1] (calibrated from run 1: median chi2=16.3)
MAX_STEPS  = 100     # optimistix iteration cap per pixel
TILE_SIZE  = 65536   # pixels per tile — match zarr chunk size (256×256); warmup uses same shape
SOLVER     = 'GaussNewton'

## 2  Load scene — Helsinki EnMAP S3

In [ ]:
config = configparser.ConfigParser()
config.read('../../config.ini')
credentials = {k: v.strip() for k, v in config['Credentials'].items()}

store = new_data_store(
    's3', max_depth=5, root='coastal-cubes/sek/',
    storage_options=dict(
        anon=False,
        key=credentials['s3_client_id'],
        secret=credentials['s3_client_secret'],
    )
)

INPUT_PREFIX  = 'helsinki/L2A_land/'
OUTPUT_PREFIX = 'helsinki/temp/'

scene_id = 'ENMAP01-____L2A-DT0000158841_20251019T101424Z_002_V010505_20260206T113145Z'

img         = store.open_data(f'{INPUT_PREFIX}{scene_id}.zarr')
scene_crs   = img.rio.crs or CRS.from_wkt(img.spatial_ref.attrs['crs_wkt'])
wavelengths = img.wavelength.values[:80]

refl  = img['reflectance'].isel(band=slice(0, 80))
rrs   = (refl.where(refl > -32768) / 10_000) / np.pi
cloud = (img['cloud'] == 1) | (img['cirrus'] == 1) | (img['haze'] == 1)
rrs   = rrs.where(~cloud)

_wc = store.open_data(f'{OUTPUT_PREFIX}{scene_id}-worldcover.zarr')
_wf = _wc['water_fraction'].values
_labeled, _ = ndi.label(_wf >= 0.5)
_sizes = np.bincount(_labeled.ravel())
_sizes[0] = 0
ocean_mask = xr.DataArray(
    np.isin(_labeled, np.where(_sizes >= 100)[0]),
    coords=_wc['water_fraction'].coords, dims=_wc['water_fraction'].dims,
)
rrs = rrs.where(ocean_mask)

Rrs_arr = rrs.transpose('y', 'x', 'band').data
n_rows, n_cols, n_obs = Rrs_arr.shape
n_water = int(ocean_mask.values.sum())   # ocean pixels (avoids full S3 read)

print(f'Image shape : {Rrs_arr.shape}')
print(f'Water pixels: {n_water}')
print(f'Wavelengths : {wavelengths[0]:.1f} – {wavelengths[-1]:.1f} nm ({n_obs} bands)')

## 3  Parameters and OE setup

Parameter set from `11_lsq_superpixel_inversion.ipynb`.
All concentrations and glint params are retrieved in **log-space** (lognormal prior,
ensures positivity without box constraints — same as NB12 but extended to glint).

`sigma_a` is calibrated so that the ±3σ interval in log-space aligns with the lmfit
`max` bound: `sigma_a = ln(max / x_a) / 3`. Concentrations use 0.7 (≈ factor 8 at
±3σ); glint widths vary per-parameter to match their respective lmfit ranges.

In [ ]:
params = lmfit.Parameters()

# --- water column — 3 free, 6 fixed at prior (not needed for glint step) ------
params.add('C_0',   value=2,   min=0, max=100, vary=True)
params.add('C_1',   value=0,   min=0, max=100, vary=False)
params.add('C_2',   value=0,   min=0, max=100, vary=False)
params.add('C_3',   value=0,   min=0, max=100, vary=False)
params.add('C_4',   value=0,   min=0, max=100, vary=False)
params.add('C_5',   value=0,   min=0, max=100, vary=False)
params.add('C_Y',   value=0.2, min=0, max=4,   vary=True)
params.add('C_X',   value=0,   min=0, max=100, vary=False)
params.add('C_Mie', value=1,   min=0, max=100, vary=True)

# --- bottom fractions (fixed — open water) ------------------------------
params.add('f_0',   value=1, min=0, max=1, vary=False)
params.add('f_1',   value=0, min=0, max=1, vary=False)
params.add('f_2',   value=0, min=0, max=1, vary=False)
params.add('f_3',   value=0, min=0, max=1, vary=False)
params.add('f_4',   value=0, min=0, max=1, vary=False)
params.add('f_5',   value=0, min=0, max=1, vary=False)

# --- bottom reflectance amplitudes (fixed) ------------------------------
params.add('B_0',   value=1/np.pi, vary=False)
params.add('B_1',   value=1/np.pi, vary=False)
params.add('B_2',   value=1/np.pi, vary=False)
params.add('B_3',   value=1/np.pi, vary=False)
params.add('B_4',   value=1/np.pi, vary=False)
params.add('B_5',   value=1/np.pi, vary=False)

# --- optical coefficients (fixed) ----------------------------------------
params.add('bb_phy_spec',         value=0.0010, vary=False)
params.add('bb_Mie_spec',         value=0.0042, vary=False)
params.add('bb_X_spec',           value=0.0086, vary=False)
params.add('a_NAP_spec_lambda_0', value=0.041,  vary=False)
params.add('S',                   value=0.014,  vary=False)
params.add('K',                   value=0,      vary=False)
params.add('S_NAP',               value=0.011,  vary=False)
params.add('n',                   value=-1,     vary=False)
params.add('lambda_0',            value=440,    vary=False)
params.add('lambda_S',            value=500,    vary=False)

# --- geometry (fixed) ---------------------------------------------------
params.add('theta_sun',  value=np.radians(30),    vary=False)
params.add('theta_view', value=np.radians(1e-10), vary=False)
params.add('n1',         value=1,                 vary=False)
params.add('n2',         value=1.33,              vary=False)
params.add('kappa_0',    value=1.0546,            vary=False)

# --- bathymetry + temperature (fixed) ------------------------------------
params.add('zB',    value=100, min=0.1, max=1000, vary=False)
params.add('T_W',   value=18,  min=0,   max=40,   vary=False)
params.add('T_W_0', value=20,                     vary=False)

# --- glint (free) --------------------------------------------------------
params.add('g_dd',  value=0.02,    min=0, max=10,  vary=True)
params.add('g_dsr', value=1/np.pi, min=0, max=10,  vary=True)
params.add('g_dsa', value=1/np.pi, min=0, max=10,  vary=True)
params.add('d_r',   value=0.01,    min=0, max=0.1, vary=True)

# --- diffuse fraction + offset (fixed) -----------------------------------
params.add('fd_d',   value=1, vary=False)
params.add('fd_s',   value=1, vary=False)
params.add('offset', value=0, min=0, max=0.01, vary=False)

# --- OE priors (log-space widths) ----------------------------------------
# sigma_a = ln(max / x_a) / 3  so that ±3σ aligns with the lmfit max bound
sigma_a = {
    'C_0':   1.30, 'C_Y':  1.00, 'C_Mie': 1.54,
    'g_dd':  2.07, 'g_dsr': 1.15, 'g_dsa': 1.15, 'd_r': 0.77,
}
log_params = ['C_0', 'C_Y', 'C_Mie', 'g_dd', 'g_dsr', 'g_dsa', 'd_r']

free = [n for n, p in params.items() if p.vary]
print(f'Free parameters ({len(free)}): {free}')

## 4  Precompute & build OE setup

In [ ]:
pre_3C = albert_mobley_3C_jax.precompute(
    wavelengths, fresh=False,
    theta_sun=float(params['theta_sun'].value),
    P=1013.25, AM=1, RH=60, H_oz=0.38, WV=2.5, alpha=1.317, beta=0.2606,
)

f_vec = albert_mobley_3C_jax.make_forward_vec(list(params.keys()), pre_3C)
setup = oe_engine.build_inversion(params, f_vec, sigma_a, log_params=log_params)

print(f'fit_names : {setup.fit_names}')
print(f'log_params: {log_params}')
print(f'n_obs     : {n_obs}')

## 5  Warm up JAX JIT

`dask_engine` does not handle warmup — JAX compiles on the first `invert_fn` call.
Pre-compile here so the timed run below pays no compilation cost.

In [ ]:
print('Warming up JAX JIT ...', end=' ', flush=True)
t_wu = time.perf_counter()
# Warmup must use TILE_SIZE — JIT is keyed on input shape, so (2, n_obs) causes
# a retrace on the first real tile.  Use prior Rrs (forward model at prior means)
# instead of zeros: the OE residual is ~0 at the prior, so the solver exits in
# 0–1 steps regardless of MAX_STEPS.  Cost = compilation only (~30–60 s).
import jax.numpy as jnp
_x_prior   = jnp.array([float(params[n].value) for n in params.keys()])
_prior_rrs = np.array(f_vec(_x_prior))                  # (n_obs,)
_dummy     = np.tile(_prior_rrs, (TILE_SIZE, 1))         # (TILE_SIZE, n_obs)
oe_engine_optx.invert_image(
    _dummy, setup, NOISE,
    solver=SOLVER, max_steps=MAX_STEPS,
    store_chi2_spectral=True, store_y_hat=True,
)
print(f'{time.perf_counter() - t_wu:.1f} s')

## 6  Run — per-pixel OE via dask tiling

`dask_engine.invert_image` tiles the image and calls `oe_engine_optx.invert_image`
per tile via the shared `invert_fn` interface.  JAX vmap runs the full tile in
one compiled call; XLA parallelises across pixels within each tile.

In [ ]:
print(f'Inverting {n_water} water pixels ({n_rows}×{n_cols} image, '
      f'{len(setup.fit_names)} free params, {n_obs} bands) ...')

t0 = time.perf_counter()
results = dask_engine.invert_image(
    Rrs_arr, setup, NOISE,
    invert_fn=oe_engine_optx.invert_image,
    tile_size=TILE_SIZE,
    solver=SOLVER,
    max_steps=MAX_STEPS,
    store_chi2_spectral=True,
    store_y_hat=True,
)
t_total = time.perf_counter() - t0

n_tiles = int(np.ceil(n_rows * n_cols / TILE_SIZE))
print(f'Done in {t_total:.1f} s  ({1000*t_total/n_water:.2f} ms/pixel, '
      f'{1000*t_total/n_tiles:.0f} ms/tile, {n_tiles} tiles)')

## 7  Noise calibration

For OE, the ideal median `chi2` is ≈ 1.
If it deviates: `NOISE_cal = NOISE × sqrt(median_chi2)` — update and re-run.

In [ ]:
chi2       = results['chi2']
valid_chi2 = chi2[np.isfinite(chi2)]
med_chi2   = float(np.median(valid_chi2))
noise_cal  = NOISE * np.sqrt(med_chi2)

print(f'NOISE = {NOISE:.4f} sr\u207b\u00b9')
print(f'Median chi2 = {med_chi2:.3f}  \u2192  calibrated NOISE = {noise_cal:.4f} sr\u207b\u00b9')
if abs(med_chi2 - 1.0) > 0.15:
    print(f'\u2192 Update NOISE = {noise_cal:.4f} and re-run.')
else:
    print('chi2 \u2248 1 \u2014 noise well-calibrated.')

## 8  Retrieved parameter maps

In [ ]:
x_hat     = results['x_hat']         # (n_rows, n_cols, n_fit)
sigma     = results['sigma']          # (n_rows, n_cols, n_fit)
A_diag    = results['A_diag']         # (n_rows, n_cols, n_fit)
H_info    = results['H_info']         # (n_rows, n_cols)
chi2_sp   = results['chi2_spectral']  # (n_rows, n_cols)
n_steps   = results['n_steps']        # (n_rows, n_cols)
fit_names = results['fit_names']

_style = {
    'C_Y':   ('YlOrBr', 0, 3,    'C_Y CDOM [1/m]'),
    'C_X':   ('Greys',  0, 50,   'C_X ISM [g/m\u00b3]'),
    'C_Mie': ('Greys',  0, 50,   'C_Mie [g/m\u00b3]'),
    'g_dd':  ('plasma', 0, 0.2,  'g_dd'),
    'g_dsr': ('plasma', 0, 0.5,  'g_dsr'),
    'g_dsa': ('plasma', 0, 0.5,  'g_dsa'),
    'd_r':   ('viridis',0, 0.05, 'd_r'),
}

n_params = len(fit_names)
ncols = 5
nrows = int(np.ceil(n_params / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.ravel() if nrows > 1 else list(axes)

for ax, name in zip(axes_flat, fit_names):
    idx = fit_names.index(name)
    if name in _style:
        cmap, vmin, vmax, title = _style[name]
    else:
        cmap, vmin, vmax, title = 'YlGn', 0, 50, f'{name} [\u00b5g/L]'
    im = ax.imshow(x_hat[..., idx], cmap=cmap, vmin=vmin, vmax=vmax, origin='upper')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

for ax in axes_flat[n_params:]:
    ax.axis('off')

plt.suptitle('albert_mobley_3C_jax OE per-pixel \u2014 retrieved parameters', fontsize=12)
plt.tight_layout()
plt.show()

## 9  OE diagnostics — chi2, H_info, A_diag, n_steps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

im0 = axes[0, 0].imshow(chi2, cmap='RdYlGn_r', origin='upper',
                         vmin=0, vmax=np.nanpercentile(valid_chi2, 98))
plt.colorbar(im0, ax=axes[0, 0], shrink=0.85)
axes[0, 0].set_title(f'chi2 (target \u2248 1, median={med_chi2:.2f})')
axes[0, 0].axis('off')

chi2_sp_v = chi2_sp[np.isfinite(chi2_sp)]
im1 = axes[0, 1].imshow(chi2_sp, cmap='RdYlGn_r', origin='upper',
                         vmin=0, vmax=np.nanpercentile(chi2_sp_v, 98))
plt.colorbar(im1, ax=axes[0, 1], shrink=0.85)
axes[0, 1].set_title(f'chi2_spectral (plain MSE, median={np.nanmedian(chi2_sp_v):.2e})')
axes[0, 1].axis('off')

H_v = H_info[np.isfinite(H_info)]
im2 = axes[0, 2].imshow(H_info, cmap='viridis', origin='upper',
                         vmin=0, vmax=np.nanpercentile(H_v, 98))
plt.colorbar(im2, ax=axes[0, 2], shrink=0.85, label='nats')
axes[0, 2].set_title(f'H_info (median={np.nanmedian(H_v):.2f} nats)')
axes[0, 2].axis('off')

im3 = axes[1, 0].imshow(A_diag.mean(axis=-1), cmap='viridis', origin='upper', vmin=0, vmax=1)
plt.colorbar(im3, ax=axes[1, 0], shrink=0.85)
axes[1, 0].set_title('mean A_diag (DFS/n_params)')
axes[1, 0].axis('off')

ns_v = n_steps[n_steps >= 0]
im4 = axes[1, 1].imshow(n_steps, cmap='plasma', origin='upper', vmin=0, vmax=MAX_STEPS)
plt.colorbar(im4, ax=axes[1, 1], shrink=0.85)
sat_pct = 100 * (ns_v == MAX_STEPS).mean()
axes[1, 1].set_title(f'n_steps (median={int(np.median(ns_v))}, '
                     f'saturated={sat_pct:.0f}%)')
axes[1, 1].axis('off')

axes[1, 2].hist(ns_v, bins=range(0, MAX_STEPS + 2), density=True,
                color='steelblue', alpha=0.8)
axes[1, 2].axvline(MAX_STEPS, color='r', ls='--', lw=1.5, label=f'cap={MAX_STEPS}')
axes[1, 2].set_xlabel('n_steps')
axes[1, 2].set_ylabel('Density')
axes[1, 2].set_title('n_steps distribution')
axes[1, 2].legend()

plt.suptitle('OE diagnostics', fontsize=12)
plt.tight_layout()
plt.show()

## 10  Posterior uncertainty maps

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.ravel() if nrows > 1 else list(axes)

for ax, name in zip(axes_flat, fit_names):
    idx = fit_names.index(name)
    s   = sigma[..., idx]
    s_v = s[np.isfinite(s)]
    im  = ax.imshow(s, cmap='Purples', origin='upper',
                    vmin=0, vmax=np.nanpercentile(s_v, 98))
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f'\u03c3({name})', fontsize=10)
    ax.axis('off')

for ax in axes_flat[n_params:]:
    ax.axis('off')

plt.suptitle('Posterior uncertainty \u03c3 per parameter', fontsize=12)
plt.tight_layout()
plt.show()

## 11  Spot-check — observed vs fitted spectra

Sample 6 pixels: best, worst, and 4 percentile-spaced by `chi2_spectral`.

In [ ]:
y_hat = results['y_hat']   # (n_rows, n_cols, n_obs)

chi2_flat  = chi2_sp.ravel()
valid_px   = np.where(np.isfinite(chi2_flat))[0]
ranked     = valid_px[np.argsort(chi2_flat[valid_px])]

picks        = [ranked[int(len(ranked) * q)] for q in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]]
labels_picks = ['best', 'p20', 'p40', 'p60', 'p80', 'worst']

Rrs_flat   = Rrs_arr.reshape(-1, n_obs)
y_hat_flat = y_hat.reshape(-1, n_obs)

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=False)
for ax, px_idx, label in zip(axes.ravel(), picks, labels_picks):
    obs    = Rrs_flat[px_idx]
    fitted = y_hat_flat[px_idx]
    chi2_px = float(chi2_flat[px_idx])

    ax.plot(wavelengths, obs,    'k-',  lw=2, label='observed')
    ax.plot(wavelengths, fitted, 'r--', lw=2, label='fitted')
    ax.fill_between(wavelengths, obs, fitted, alpha=0.15, color='r')
    ax.set_title(f'{label} \u2014 chi2_sp={chi2_px:.2e}')
    ax.set_xlabel('\u03bb [nm]')
    ax.set_ylabel('Rrs [sr\u207b\u00b9]')
    if label == 'best':
        ax.legend(fontsize=8)

plt.suptitle('Observed vs fitted spectra', fontsize=12)
plt.tight_layout()
plt.show()

## 12  Save to S3 store (optional)

In [ ]:
# ds = dask_engine.to_dataset(
#     results, wavelengths=wavelengths,
#     coords={'y': rrs.y.values, 'x': rrs.x.values},
# ).rio.write_crs(scene_crs)
# store.write_data(ds, f'{OUTPUT_PREFIX}{scene_id}-glint_wq_params_oe_dask_am3c.zarr', replace=True)
# print('Saved.')

## 13  Glint correction

`albert_mobley_3C.forward = albert_mobley.forward + surface_reflectance.forward`

The glint term is strictly additive and can be computed directly via
`reflectance_jax.make_forward_vec` — no need to call the full coupled model twice.
This halves the cost relative to a subtraction approach and uses the same precomputed
atmospheric arrays already in memory.

    glint    = surface_reflectance_jax(x̂_glint)      # direct
    Rrs_corr = Rrs_obs − glint

In [ ]:
import jax.numpy as jnp
import dask
import dask.array as da
from bio_optics.surface import reflectance_jax

# Precompute atmospheric arrays for the glint model (Mode A: theta_sun baked in)
pre_surf = reflectance_jax.precompute(
    wavelengths, theta_sun=float(params['theta_sun'].value),
    P=1013.25, AM=1, RH=60, H_oz=0.38, WV=2.5, alpha=1.317, beta=0.2606,
)

surf_param_names = ['fd_d', 'fd_s', 'g_dd', 'g_dsr', 'g_dsa', 'theta_view', 'n1', 'n2', 'd_r']
glint_free_names = ['g_dd', 'g_dsr', 'g_dsa', 'd_r']
n_surf = len(surf_param_names)

f_glint_vec  = reflectance_jax.make_forward_vec(surf_param_names, pre_surf)
f_glint_vmap = jax.jit(jax.vmap(f_glint_vec))
# Warm up with TILE_SIZE to match actual tile shape — avoids retrace on first tile.
_ = f_glint_vmap(jnp.zeros((TILE_SIZE, n_surf), dtype=jnp.float64))
print('JAX warmup done')

surf_base         = np.array([float(params[n].value) for n in surf_param_names])
glint_idx_in_surf = np.array([surf_param_names.index(n) for n in glint_free_names])
glint_idx_in_fit  = np.array([fit_names.index(n)        for n in glint_free_names])

n_pixels   = n_rows * n_cols
Rrs_flat   = Rrs_arr.reshape(-1, n_obs)              # dask
x_hat_flat = x_hat.reshape(-1, x_hat.shape[-1])      # numpy

def _correct_tile(Rrs_tile, x_hat_tile):
    """Compute glint inline and return glint-corrected Rrs for one tile."""
    tile_params = np.broadcast_to(surf_base, (len(Rrs_tile), n_surf)).copy()
    tile_params[:, glint_idx_in_surf] = x_hat_tile[:, glint_idx_in_fit]
    glint = np.array(f_glint_vmap(jnp.array(tile_params, dtype=jnp.float64)))
    return np.array(Rrs_tile) - glint

starts = range(0, n_pixels, TILE_SIZE)
ends   = [*range(TILE_SIZE, n_pixels, TILE_SIZE), n_pixels]

corr_tiles = [
    da.from_delayed(
        dask.delayed(_correct_tile)(Rrs_flat[s:e], x_hat_flat[s:e]),
        shape=(e - s, n_obs), dtype=np.float64,
    )
    for s, e in zip(starts, ends)
]
Rrs_corr = da.concatenate(corr_tiles).reshape(n_rows, n_cols, n_obs)
water_mask = ocean_mask.values   # numpy bool (n_rows, n_cols), from §2
print(f'Rrs_corr graph built — {len(corr_tiles)} tiles, lazy until write')

In [ ]:
Rrs_flat      = Rrs_arr.reshape(-1, n_obs)    # dask; matplotlib auto-computes per pixel
Rrs_corr_flat = Rrs_corr.reshape(-1, n_obs)   # dask; same

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, px_idx, label in zip(axes.ravel(), picks, labels_picks):
    ax.plot(wavelengths, Rrs_flat[px_idx],      'k-',  lw=2,   label='original')
    ax.plot(wavelengths, Rrs_corr_flat[px_idx], 'b-',  lw=1.5, label='corrected')
    ax.fill_between(wavelengths, Rrs_corr_flat[px_idx], Rrs_flat[px_idx],
                    alpha=0.25, color='orange', label='glint')
    ax.set_title(label)
    ax.set_xlabel('λ [nm]')
    ax.set_ylabel('Rrs [sr⁻¹]')
    if label == 'best':
        ax.legend(fontsize=8)

plt.suptitle('Glint correction — spot-check pixels (§11 ranking)', fontsize=12)
plt.tight_layout()
plt.show()

## 14  Save glint-corrected Rrs (optional)

In [ ]:
# Rrs_corr_da = xr.DataArray(
#     Rrs_corr.transpose(2, 0, 1).astype(np.float32),
#     dims=['band', 'y', 'x'],
#     coords={'band': np.arange(1, n_obs + 1), 'wavelength': ('band', wavelengths),
#             'y': rrs.y.values, 'x': rrs.x.values},
#     attrs={'long_name': 'Glint-corrected Rrs', 'units': 'sr-1'},
# ).rio.write_crs(scene_crs)
# store.write_data(Rrs_corr_da.to_dataset(name='Rrs_corr'),
#                  f'{OUTPUT_PREFIX}{scene_id}-Rrs_glint_corrected.zarr', replace=True)
# print('Saved.')